# Pruebas Chi-cuadrado y Fisher para variables categóricas (parte 2)

Supongamos que como Científicos de Datos en un e-commerce y queremos determinar si el color del botón en el sitio web afecta la decisión de compra de los usuarios. Así que presentamos el botón azul “Compra ya” a 20 usuarios y el botón verde “Compra ya” a otros 20 usuarios.

**¿La tasa de conversión depende del color del botón?**

## 1. Desarrollo de la prueba de hipótesis

- En este caso nuestras variables a comparar son categóricas:
    - Color del botón: verde o azul
    - Decisión de compra: comprar o no comprar
- Y además tendremos una variable numérica: la tasa de conversión (proporción de personas que compró con respecto al número total de personas)

### 1.1. Paso 1: definir el problema del negocio

> **¿Hay un color de botón que genera más compras que otro?**

### 1.2. Paso 2: redactar el problema del negocio como un problema de Ciencia de Datos/Machine Learning

> **¿Existen alguna relación, desde el punto de vista estadístico, entre el color del botón y la decisión de compra?**

### 1.3. Paso 3: definir $H_0$ y $H_1$

- $H_0$: el color del botón y la decisión de comprar son **independientes**
- $H_1$: **hay relación** entre el color del botón y la decisión de comprar

### 1.4. Paso 4: definir $\alpha$

Asumiremos un nivel de significancia $\alpha = 0.05$.

### 1.5. Paso 5: definir la potencia de la prueba ($1-\beta$) y el tamaño de las muestras ($n$)

La lógica para estimar el tamaño de la muestra es similar a lo que ya hemos visto en lecciones anteriores y en el curso de Estadística Inferencial Paramétrica. En este caso la diferencia es que chi-cuadrado tiene su propio conjunto de ecuaciones (no es necesario realizar aproximaciones como en casos anteriores).

Este es el paso a paso:

1. Definimos el tamaño del efecto. En el caso de Chi-cuadrado el tamaño de efecto se define como:


$$\omega = \sqrt{\frac{\chi_2}{n}}$$

donde $n$ es el número de observaciones.

Y esta es la manera de interpretarlo:

- $\omega \leq 0.1 \rightarrow \text{efecto "pequeño"}$
- $\omega \leq 0.3 \rightarrow \text{efecto "mediano"}$
- $\omega > 0.5 \rightarrow \text{efecto "grande"}$

**Asumiremos un tamaño del efecto de 0.4**

2. Definir los grados de libertad a partir de la tabla de contingencia:

$$df = (R-1)x(C-1)$$

donde $R$ es el número de filas de la tabla y $C$ el número de columnas.

3. Definir la potencia de la prueba

**Asumiremos una potencia de 0.9**

Y con esta información (junto con el nivel de significancia definido anteriormente) 

In [1]:
from statsmodels.stats.power import GofChisquarePower
import math

# Diseño del experimento
w = 0.4     # Tamaño del efecto
alpha = 0.05 # Nivel de significancia
power = 0.9  # Potencia de la prueba
df = (2-1)*(2-1) # tenemos R = 2 (colores de botón) y C=2 (compró/no compró)

# Instancia de FTestAnovaPower
analisis = GofChisquarePower()

# Cálculo del número total de datos en todo el experimento (sin discriminar A o B)
n = analisis.solve_power(
    effect_size=w,
    alpha=alpha, 
    power=power, 
    n_bins= df+1 
)
n = math.ceil(n)

print(f"Tamaño del efecto esperado: {w}")
print(f"Tamaño total de la muestra: {n}")
print(f"Lo que equivale a aproximadamente {n/2} datos por grupo (A, B)")

Tamaño del efecto esperado: 0.4
Tamaño total de la muestra: 66
Lo que equivale a aproximadamente 33.0 datos por grupo (A, B)


Es decir que:

> **Debemos presentar cada sitio web (A o B) a al menos 33 personas para tener un tamaño del efecto esperado de 0.4 (mediano-grande), con un nivel de significancia de 0.05 y una potencia de la prueba esperada de 0.9**

### 1.6. Paso 6: recolectar y preparar los datos

Supondremos que ya hemos recolectado los datos. Estos se encuentran en el dataset "dataset_chi_cuadrado.csv":

In [2]:
import pandas as pd

RUTA = '/Users/miguel/Library/CloudStorage/GoogleDrive-miguel@codificandobits.com/My Drive/02-CODIFICANDOBITS.COM/04-Academia/01-Cursos/46-2026-04-EstadisticaInferencialNoParametrica/data/'

datos = pd.read_csv(RUTA + 'dataset_chi_cuadrado.csv')
datos

,user_id,grupo,compro
0,470,A,1
1,576,A,1
2,1899,B,0
3,362,A,0
4,168,A,0
...,...,...,...
2501,2195,B,0
2502,840,A,0
2503,881,A,0
2504,2385,B,0


En total tenemos 2506 registros. Comencemos eliminando la columna "user_id" (no es necesaria) y verificando cuántos datos tenemos en cada agrupación:

In [3]:
datos.drop(columns='user_id', inplace=True)
datos['grupo'].value_counts()

grupo
A    1253
B    1253
Name: count, dtype: int64

Perfecto, el dataset está perfectamente equilibrado (aunque esto no es un requisito para la prueba).

La columna "compró" indica si el usuario realizó una compra (1) o no (0). Podemos calcular la tasa de conversión simplemente promediando esta columna para cada agrupación:

In [4]:
datos.groupby('grupo')['compro'].mean()

grupo
A    0.104549
B    0.172386
Name: compro, dtype: float64

Y en principio vemos que la tasa de conversión en B (17.2%) es bastante más grande que la tasa de conversión que se obtiene con A (10.5%).

Así que la idea es verificar si estas diferencias son estadísticamente significativas.

**También es importante notar que la columna "compró" realmente es categórica (0, 1) y no numérica** lo que justifica precisamente el uso de la prueba Chi-cuadrado.

### 1.7. Paso 7: aplicar la prueba estadística para obtener el valor p

En este caso podemos usar Scipy (chi2_contingency) o "pingouin". Veamos cómo usar "pingouin":

In [5]:
from pingouin import chi2_independence

esperado, observado, resultados = chi2_independence(datos, x='grupo', y='compro')

En este caso la prueba nos arroja la tabla de contingencia de valores esperados ("esperado") y con las observaciones ("observado") así como los resultados de la prueba.

Veamos la tabla "esperado":

In [6]:
esperado

compro,0,1
grupo,,
A,1079.5,173.5
B,1079.5,173.5


Vemos que esta tabla contiene la misma cantidad de compradores y no compradores para cada grupo (pues asume que la hipótesis nula es cierta).

Veamos la tabla calculada directamente a partir de las observaciones:

In [7]:
observado

compro,0,1
grupo,,
A,1121.5,131.5
B,1037.5,215.5


Acá hay una diferencia notoria, pues en el grupo A hubo 131.5 compradores mientras que en B 215.5 lo que evidencia que al parecer B tiene mayores tasas de conversión que A.

Veamos qué nos dice la prueba estadística:

In [8]:
resultados

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.000000,23.602474,1.0,1.184352e-06,0.097048,0.998124
1,cressie-read,0.666667,23.647477,1.0,1.156975e-06,0.097141,0.998151
2,log-likelihood,0.000000,23.806705,1.0,1.065108e-06,0.097467,0.998246
3,freeman-tukey,-0.500000,23.988298,1.0,9.692301e-07,0.097838,0.998347
4,mod-log-likelihood,-1.000000,24.225097,1.0,8.570814e-07,0.098320,0.998471
5,neyman,-2.000000,24.873198,1.0,6.122779e-07,0.099627,0.998766


Acá nos interesa sólo la primera fila, que contiene el cálculo de la estadística $\chi_2$ clásico (el que vimos en el video anterior). Las otras filas contienen variantes de ese cálculo.

En esta fila vemos que:

- $\chi2$ (la estadística de la prueba) es 23.6
- El valor p ($pval$) es 1.18e-06 (mucho menor que 0.05)
- "cramer" es otra manera de reportar el tamaño del efecto ($V$). La equivalencia con $\omega$ es:

$$\omega = V\sqrt(k-1)$$ 

donde $k$ es el mínimo entre las filas y columnas de la tabla de contingencia (en este caso este mínimo es 2.

Así que:

$$\omega = 0.097048*\sqrt{1} = 0.097048$$
- "power" es la potencia post-hoc de la prueba (0.99)

### 1.8. Paso 8: aceptar o rechazar $H_0$

Con la prueba anterior y el valor p de 1.184352e-06 podemos rechazar la hipótesis nula (A y B tienen la misma tasa de conversión) y por tanto nos inclinamos por la hipótesis alterna:

> Existen diferencias estadísticamente significativas en las tasas de conversión entre A y B

**Recordemos que la prueba no nos indica en qué dirección están las diferencias**. Pero esto podemos determinarlo directamente a partir de los datos, pues vimos que las tasas de conversión son:

- Del 10.5% para el grupo A
- Y del 17.2% para el grupo B

Así que en resumen desde el punto de vista estadístico **B genera tasas de conversión MAYORES que A**.

### 1.9. Paso 9: evaluar el tamaño del efecto y la potencia de la prueba actualizadas

El tamaño del efecto que obtuvimos hace un momento es $\omega = 0.097$ que es un tamaño "pequeño".

Es decir que, a pesar de las apariencias, aunque B tiene una tasa de conversión casi 7 puntos porcentuales mayor que A, realmente desde el punto de vista estadístico esto no es una gran diferencia... **!Aunque sí podría serlo desde el punto de vista del negocio!**

Por otra parte, la potencia post-hoc es de 0.99. Es decir que podemos estar 99% confiados de que realmente las diferencias entre A y B están presentes, según los datos que hemos recolectado.